# AIC 2026 - Kaggle frame extraction smoke

Attach these datasets before running:

- `lyduchoang/aic-26-video` for raw videos
- `khoalequangminh/aic-test-dataset` for metadata/map files

This notebook clones/pulls the repo branch, runs the offline frame extraction CLI for one video, then runs TransNetV2 shot detection. It does not run DAM, OCR, ASR, embeddings, or tests.

## Optional fast paths

The setup cell below is preconfigured for `L21_V001` and tries both common Kaggle mount styles:

- `/kaggle/input/datasets/<owner>/<slug>/...`
- `/kaggle/input/<slug>/...`

When `AIC_VIDEO_PATH` resolves, the notebook skips the full `/kaggle/input` scan. `AIC_MAP_CSV` and `AIC_MEDIA_INFO` are optional; if the map CSV is missing, smoke extraction falls back to fixed timestamps. TransNetV2 runs with the PyTorch backend by default. It uses Kaggle's PyTorch runtime and downloads a commit-pinned `.pth` checkpoint when no local checkpoint has been attached.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    token = None

if token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}",
    })

if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repo: {TARGET}")
if not TARGET.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)

os.chdir(TARGET)
print("Repo:", Path.cwd())
run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=TARGET)
run(["git", "rev-parse", "--short", "HEAD"], cwd=TARGET)


In [ ]:
from collections import Counter
from pathlib import Path
import os
import subprocess

os.environ["AIC_DATA_ROOT"] = "/kaggle/input"
os.environ["AIC_ARTIFACT_ROOT"] = "/kaggle/working/aic2026-artifacts"
os.environ["AIC_VIDEO_ID"] = "L21_V001"

VIDEO_PATH_CANDIDATES = [
    "/kaggle/input/datasets/lyduchoang/aic-26-video/Video/Video/Videos_L21_a/video/L21_V001.mp4",
    "/kaggle/input/aic-26-video/Video/Video/Videos_L21_a/video/L21_V001.mp4",
    "/kaggle/input/aic-26-video/Video/Videos_L21_a/video/L21_V001.mp4",
]
MAP_CSV_CANDIDATES = [
    "/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes/L21_V001.csv",
    "/kaggle/input/aic-test-dataset/data/map-keyframes/L21_V001.csv",
    "/kaggle/input/aic-test-dataset/map-keyframes/L21_V001.csv",
]
MEDIA_INFO_CANDIDATES = [
    "/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/media-info/L21_V001.json",
    "/kaggle/input/aic-test-dataset/data/media-info/L21_V001.json",
    "/kaggle/input/aic-test-dataset/media-info/L21_V001.json",
]
TRANSNETV2_ENTRYPOINT_CANDIDATES = [
    "/kaggle/input/transnetv2/inference/transnetv2.py",
    "/kaggle/input/transnetv2/TransNetV2/inference/transnetv2.py",
    "/kaggle/input/transnet-v2/inference/transnetv2.py",
    "/kaggle/input/transnet-v2/TransNetV2/inference/transnetv2.py",
    "/kaggle/working/TransNetV2/inference/transnetv2.py",
]
TRANSNETV2_WEIGHTS_CANDIDATES = [
    "/kaggle/input/transnetv2-weights/inference/transnetv2-weights",
    "/kaggle/input/transnetv2-weights/transnetv2-weights",
    "/kaggle/input/transnetv2-weights",
    "/kaggle/input/transnetv2/inference/transnetv2-weights",
    "/kaggle/input/transnetv2/TransNetV2/inference/transnetv2-weights",
    "/kaggle/input/transnet-v2/inference/transnetv2-weights",
]

def resolve_path_env(name: str, candidates: list[str], *, required: bool) -> str:
    current = os.environ.get(name, "").strip()
    if current:
        return current
    for candidate in candidates:
        if Path(candidate).is_file():
            return candidate
    if required:
        return candidates[0]
    return ""

def resolve_dir_env(name: str, candidates: list[str]) -> str:
    current = os.environ.get(name, "").strip()
    if current:
        return current
    for candidate in candidates:
        if Path(candidate).is_dir():
            return candidate
    return ""

def ensure_transnetv2_entrypoint() -> str:
    current = os.environ.get("AIC_TRANSNETV2_ENTRYPOINT", "").strip()
    if current and Path(current).is_file():
        return current
    if current:
        print(f"WARNING: AIC_TRANSNETV2_ENTRYPOINT is set but not a file: {current}", flush=True)
    for candidate in TRANSNETV2_ENTRYPOINT_CANDIDATES:
        if Path(candidate).is_file():
            return candidate

    repo_dir = Path("/kaggle/working/TransNetV2-source")
    entrypoint = repo_dir / "inference" / "transnetv2.py"
    print("TransNetV2 source not found in /kaggle/input; cloning source only (Git LFS disabled)...", flush=True)
    try:
        if not repo_dir.exists():
            clone_env = os.environ.copy()
            clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
            subprocess.run(
                ["git", "clone", "--depth=1", "https://github.com/soCzech/TransNetV2.git", str(repo_dir)],
                check=True,
                env=clone_env,
            )
        else:
            subprocess.run(["git", "fetch", "origin", "master"], cwd=repo_dir, check=False)
    except subprocess.CalledProcessError as error:
        print(f"WARNING: failed to clone/update TransNetV2: {error}", flush=True)

    return str(entrypoint) if entrypoint.is_file() else ""

os.environ["AIC_VIDEO_PATH"] = resolve_path_env("AIC_VIDEO_PATH", VIDEO_PATH_CANDIDATES, required=True)
os.environ["AIC_MAP_CSV"] = resolve_path_env("AIC_MAP_CSV", MAP_CSV_CANDIDATES, required=False)
os.environ["AIC_MEDIA_INFO"] = resolve_path_env("AIC_MEDIA_INFO", MEDIA_INFO_CANDIDATES, required=False)
os.environ["AIC_DISCOVER_SUPPORT_FILES"] = "0"
os.environ["AIC_RUN_TRANSNETV2"] = "1"
os.environ["AIC_TRANSNETV2_BACKEND"] = "pytorch"
os.environ["AIC_TRANSNETV2_ENTRYPOINT"] = ensure_transnetv2_entrypoint()
if os.environ["AIC_TRANSNETV2_BACKEND"] == "tensorflow":
    os.environ["AIC_TRANSNETV2_WEIGHTS"] = resolve_dir_env("AIC_TRANSNETV2_WEIGHTS", TRANSNETV2_WEIGHTS_CANDIDATES)
else:
    os.environ["AIC_TRANSNETV2_WEIGHTS"] = os.environ.get("AIC_TRANSNETV2_WEIGHTS", "").strip()

def log(*args):
    print(*args, flush=True)

input_root = Path(os.environ["AIC_DATA_ROOT"])
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
explicit_map_csv = os.environ.get("AIC_MAP_CSV", "").strip()
explicit_media_info = os.environ.get("AIC_MEDIA_INFO", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"
log("AIC_DATA_ROOT:", input_root)
log("AIC_ARTIFACT_ROOT:", os.environ["AIC_ARTIFACT_ROOT"])
log("AIC_VIDEO_ID:", os.environ["AIC_VIDEO_ID"])
log("AIC_VIDEO_PATH:", explicit_video_path or "<auto-discover>")
log("AIC_MAP_CSV:", explicit_map_csv or "<auto/fallback>")
log("AIC_MEDIA_INFO:", explicit_media_info or "<auto/none>")
log("AIC_DISCOVER_SUPPORT_FILES:", discover_support_files)

log("\nGPU preflight:")
try:
    gpu_report = subprocess.run(
        ["nvidia-smi", "-L"],
        check=False,
        capture_output=True,
        text=True,
    )
    log(gpu_report.stdout.strip() or gpu_report.stderr.strip() or "nvidia-smi returned no output")
    if gpu_report.returncode == 0:
        gpu_lines = [line for line in gpu_report.stdout.splitlines() if line.strip()]
        t4_lines = [line for line in gpu_lines if "T4" in line.upper()]
        if len(t4_lines) != 2:
            log(f"WARNING: expected Kaggle GPU T4 x2, detected {len(t4_lines)} T4 GPU(s).")
except FileNotFoundError:
    log("WARNING: nvidia-smi is not available. Check Kaggle Settings > Accelerator = GPU T4 x2.")

def print_tree(root: Path, *, max_depth: int = 3, max_items: int = 200):
    log("\nInput tree preview:")
    shown = 0

    def walk(directory: Path, depth: int):
        nonlocal shown
        if shown >= max_items or depth > max_depth:
            return
        try:
            children = sorted(directory.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
        except PermissionError as error:
            log("  " * (depth - 1) + f"<permission denied: {error}>")
            return
        for path in children:
            if shown >= max_items:
                return
            rel = path.relative_to(root)
            indent = "  " * (len(rel.parts) - 1)
            suffix = "/" if path.is_dir() else ""
            log(f"{indent}{rel.name}{suffix}")
            shown += 1
            if path.is_dir():
                walk(path, depth + 1)

    walk(root, 1)
    if shown >= max_items:
        log(f"... truncated after {max_items} items")

print_tree(input_root)

if explicit_video_path:
    log("\nExplicit AIC_VIDEO_PATH is set; skipping full /kaggle/input scan.")
    for label, raw_path in [
        ("video", explicit_video_path),
        ("map_csv", explicit_map_csv),
        ("media_info", explicit_media_info),
    ]:
        if raw_path:
            path = Path(raw_path)
            log(f"  {label}: {path} exists={path.exists()} is_file={path.is_file()}")
    if not Path(explicit_video_path).is_file():
        log("\nVideo path candidates tried:")
        for candidate in VIDEO_PATH_CANDIDATES:
            path = Path(candidate)
            log(f"  {path} exists={path.exists()} is_file={path.is_file()}")
        raise FileNotFoundError(f"AIC_VIDEO_PATH does not exist: {explicit_video_path}")
    if not explicit_map_csv:
        log("  map_csv not provided; smoke extraction will use fallback timestamps unless AIC_DISCOVER_SUPPORT_FILES=1.")
else:
    log("\nScanning /kaggle/input for suffix counts and first video candidates...")
    suffix_counts = Counter()
    video_suffixes = {".avi", ".mkv", ".mov", ".mp4", ".webm"}
    video_candidates = []
    scanned_files = 0
    for path in input_root.rglob("*"):
        if not path.is_file():
            continue
        scanned_files += 1
        suffix = path.suffix.lower() or "<none>"
        suffix_counts[suffix] += 1
        if suffix in video_suffixes and len(video_candidates) < 20:
            video_candidates.append(path)
        if scanned_files % 1000 == 0:
            log(f"  scanned_files={scanned_files}")

    log(f"Scan complete. files={scanned_files}")
    log("Suffix counts:")
    for suffix, count in suffix_counts.most_common(20):
        log(f"  {suffix}: {count}")

    log("First video candidates:")
    for path in video_candidates:
        log(" ", path)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
video_id = os.environ["AIC_VIDEO_ID"]
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
explicit_map_csv = os.environ.get("AIC_MAP_CSV", "").strip()
explicit_media_info = os.environ.get("AIC_MEDIA_INFO", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"
command = [
    "python",
    "scripts/extract_frame_samples.py",
    "--config",
    "configs/offline/frame_extraction.yaml",
    "--video-id",
    video_id,
    "--output-root",
    str(artifact_root),
    "--limit",
    "10",
    "--resume",
]
if explicit_video_path:
    command.extend(["--video-path", explicit_video_path])
    if discover_support_files:
        command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
else:
    command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
if explicit_map_csv:
    command.extend(["--map-csv", explicit_map_csv])
if explicit_media_info:
    command.extend(["--media-info", explicit_media_info])
print("$", " ".join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
lines = []
for line in process.stdout:
    print(line, end="", flush=True)
    lines.append(line.strip())
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Frame extraction CLI failed with code {return_code}")

report = None
for line in reversed([line for line in lines if line]):
    try:
        report = json.loads(line)
        break
    except json.JSONDecodeError:
        pass
if report is None:
    raise RuntimeError("Frame extraction CLI did not print a JSON report")
manifest_path = Path(report["output"])
print("Manifest:", manifest_path)
print("Frames:", report.get("frames"))


## TransNetV2 shot detection

This cell runs by default. Attach a TransNetV2 runtime/weights dataset and set `AIC_TRANSNETV2_ENTRYPOINT` if the auto-detected path is not correct.

- `AIC_RUN_TRANSNETV2=1` runs detection.
- `AIC_RUN_TRANSNETV2=0` skips detection only if you intentionally want frame smoke without shots.
- `AIC_TRANSNETV2_SCENES_FILE` can be used to parse a precomputed scenes file instead of running inference.

Logs are streamed live from the CLI and TransNetV2 process.

In [ ]:
import importlib.metadata
import importlib.util
import hashlib
import json
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

RUN_TRANSNETV2 = os.environ.get("AIC_RUN_TRANSNETV2", "1") == "1"
TRANSNETV2_BACKEND = os.environ.get("AIC_TRANSNETV2_BACKEND", "pytorch").strip().lower()
transnetv2_entrypoint_raw = os.environ.get("AIC_TRANSNETV2_ENTRYPOINT", "").strip()
TRANSNETV2_ENTRYPOINT = Path(transnetv2_entrypoint_raw or "/kaggle/input/transnetv2/inference/transnetv2.py")
TRANSNETV2_WEIGHTS = os.environ.get("AIC_TRANSNETV2_WEIGHTS")
TRANSNETV2_SCENES_FILE = os.environ.get("AIC_TRANSNETV2_SCENES_FILE")
PYTORCH_MODULE = Path(os.environ.get("AIC_TRANSNETV2_PYTORCH_MODULE", "/kaggle/working/TransNetV2-source/inference-pytorch/transnetv2_pytorch.py"))
PYTORCH_WEIGHTS_URL = "https://huggingface.co/ByteDance/shot2story/resolve/ff853c571fd92eb4e0c5713e27f2a323ac903f67/transnetv2-pytorch-weights.pth?download=true"
PYTORCH_WEIGHTS_SHA256 = "a313d0b3bebfa9a71914b375bfdf918a30b5c3b1e6be51972d35dd8078b442de"
explicit_video_path = os.environ.get("AIC_VIDEO_PATH", "").strip()
discover_support_files = os.environ.get("AIC_DISCOVER_SUPPORT_FILES", "0") == "1"
TRANSNETV2_ENTRYPOINT_CANDIDATES = globals().get("TRANSNETV2_ENTRYPOINT_CANDIDATES", [
    "/kaggle/input/transnetv2/inference/transnetv2.py",
    "/kaggle/input/transnetv2/TransNetV2/inference/transnetv2.py",
    "/kaggle/input/transnet-v2/inference/transnetv2.py",
    "/kaggle/input/transnet-v2/TransNetV2/inference/transnetv2.py",
    "/kaggle/working/TransNetV2/inference/transnetv2.py",
    "/kaggle/working/TransNetV2-source/inference/transnetv2.py",
])
if TRANSNETV2_BACKEND == "pytorch":
    TRANSNETV2_ENTRYPOINT = PYTORCH_MODULE
    TRANSNETV2_ENTRYPOINT_CANDIDATES.append(str(PYTORCH_MODULE))

print("RUN_TRANSNETV2:", RUN_TRANSNETV2, flush=True)
print("TRANSNETV2_BACKEND:", TRANSNETV2_BACKEND, flush=True)
print("TRANSNETV2_ENTRYPOINT:", TRANSNETV2_ENTRYPOINT, flush=True)
print("TRANSNETV2_WEIGHTS:", TRANSNETV2_WEIGHTS, flush=True)
print("TRANSNETV2_SCENES_FILE:", TRANSNETV2_SCENES_FILE, flush=True)

def describe_module(module_name: str) -> bool:
    spec = importlib.util.find_spec(module_name)
    if spec is None:
        print(f"RUNTIME {module_name}: MISSING", flush=True)
        return False
    try:
        version = importlib.metadata.version(module_name)
    except importlib.metadata.PackageNotFoundError:
        version = "installed (version unavailable)"
    print(f"RUNTIME {module_name}: {version}", flush=True)
    return True

def ensure_ffmpeg_python() -> None:
    if describe_module("ffmpeg"):
        return
    print("Installing lightweight ffmpeg-python dependency...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ffmpeg-python"], check=True)
    if not describe_module("ffmpeg"):
        raise ModuleNotFoundError("ffmpeg-python installation completed but module ffmpeg is still unavailable.")

def is_lfs_pointer(path: Path) -> bool:
    if not path.is_file() or path.stat().st_size > 1024:
        return False
    with path.open("rb") as stream:
        return stream.read(64).startswith(b"version https://git-lfs.github.com/spec/v1")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def ensure_pytorch_checkpoint() -> Path:
    target = Path(TRANSNETV2_WEIGHTS) if TRANSNETV2_WEIGHTS else Path("/kaggle/working/transnetv2-pytorch/transnetv2-pytorch-weights.pth")
    if TRANSNETV2_WEIGHTS and not target.is_file():
        raise FileNotFoundError(f"AIC_TRANSNETV2_WEIGHTS must point to a .pth file: {target}")
    if target.is_file() and sha256_file(target) == PYTORCH_WEIGHTS_SHA256:
        print(f"TRANSNETV2_PYTORCH_WEIGHTS: using {target}", flush=True)
        return target
    if TRANSNETV2_WEIGHTS:
        raise ValueError(f"AIC_TRANSNETV2_WEIGHTS failed SHA-256 verification: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(".partial")
    print(f"Downloading pinned PyTorch checkpoint to {target}...", flush=True)
    urllib.request.urlretrieve(PYTORCH_WEIGHTS_URL, partial)
    actual_sha256 = sha256_file(partial)
    if actual_sha256 != PYTORCH_WEIGHTS_SHA256:
        raise ValueError(f"Downloaded PyTorch checkpoint SHA-256 mismatch: {actual_sha256}")
    os.replace(partial, target)
    print(f"TRANSNETV2_PYTORCH_WEIGHTS: downloaded {target}", flush=True)
    return target

if not TRANSNETV2_SCENES_FILE:
    if TRANSNETV2_BACKEND == "pytorch":
        if not describe_module("torch"):
            raise ModuleNotFoundError("PyTorch is required for the default TransNetV2 backend.")
        TRANSNETV2_WEIGHTS = str(ensure_pytorch_checkpoint())
        os.environ["AIC_TRANSNETV2_WEIGHTS"] = TRANSNETV2_WEIGHTS
    elif TRANSNETV2_BACKEND == "tensorflow":
        describe_module("tensorflow")
        resolved_weights = Path(TRANSNETV2_WEIGHTS) if TRANSNETV2_WEIGHTS else TRANSNETV2_ENTRYPOINT.parent / "transnetv2-weights"
        required_weight_files = [
            resolved_weights / "saved_model.pb",
            resolved_weights / "variables" / "variables.index",
            resolved_weights / "variables" / "variables.data-00000-of-00001",
        ]
        missing_weight_files = [str(path) for path in required_weight_files if not path.is_file()]
        lfs_pointer_files = [str(path) for path in required_weight_files if is_lfs_pointer(path)]
        print(f"TRANSNETV2_RESOLVED_WEIGHTS: {resolved_weights} exists={resolved_weights.is_dir()}", flush=True)
        print(f"TRANSNETV2_MISSING_WEIGHT_FILES: {missing_weight_files or '<none>'}", flush=True)
        print(f"TRANSNETV2_LFS_POINTER_FILES: {lfs_pointer_files or '<none>'}", flush=True)
    else:
        raise ValueError(f"Unsupported AIC_TRANSNETV2_BACKEND: {TRANSNETV2_BACKEND}")

if not RUN_TRANSNETV2:
    print("Skipping TransNetV2 because AIC_RUN_TRANSNETV2=0.", flush=True)
else:
    if not TRANSNETV2_SCENES_FILE and not TRANSNETV2_ENTRYPOINT.is_file():
        print("\nTransNetV2 entrypoint candidates tried:", flush=True)
        for candidate in TRANSNETV2_ENTRYPOINT_CANDIDATES:
            path = Path(candidate)
            print(f"  {path} exists={path.exists()} is_file={path.is_file()}", flush=True)
        environment_name = "AIC_TRANSNETV2_PYTORCH_MODULE" if TRANSNETV2_BACKEND == "pytorch" else "AIC_TRANSNETV2_ENTRYPOINT"
        raise FileNotFoundError(f"TransNetV2 entrypoint does not exist; set {environment_name} to the correct path.")
    if TRANSNETV2_BACKEND == "tensorflow" and not TRANSNETV2_SCENES_FILE and (missing_weight_files or lfs_pointer_files):
        raise FileNotFoundError(
            "TransNetV2 weights are unavailable. Attach a Kaggle Input dataset containing "
            "transnetv2-weights/{saved_model.pb,variables/variables.index,variables/variables.data-00000-of-00001} "
            "and set AIC_TRANSNETV2_WEIGHTS to that directory. The source-clone files are Git LFS pointers, not model weights."
        )
    if TRANSNETV2_BACKEND == "tensorflow" and not TRANSNETV2_SCENES_FILE:
        ensure_ffmpeg_python()
    command = [
        "python",
        "-u",
        "scripts/run_transnetv2_shots.py",
        "--config",
        "configs/offline/frame_extraction.yaml",
        "--backend",
        TRANSNETV2_BACKEND,
        "--video-id",
        os.environ["AIC_VIDEO_ID"],
        "--output-root",
        os.environ["AIC_ARTIFACT_ROOT"],
        "--resume",
    ]
    if explicit_video_path:
        command.extend(["--video-path", explicit_video_path])
        if discover_support_files:
            command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
    else:
        command.extend(["--search-root", os.environ["AIC_DATA_ROOT"]])
    if TRANSNETV2_SCENES_FILE:
        command.extend(["--scenes-file", TRANSNETV2_SCENES_FILE])
    else:
        command.extend(["--entrypoint", str(TRANSNETV2_ENTRYPOINT)])
        if TRANSNETV2_WEIGHTS:
            command.extend(["--weights", TRANSNETV2_WEIGHTS])

    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line.strip())
    return_code = process.wait()
    if return_code != 0:
        tail = "\n".join(line for line in lines[-80:] if line) or "<no CLI output captured>"
        raise RuntimeError(
            f"TransNetV2 shot detection failed with code {return_code}. Last output:\n{tail}"
        )

    shot_report = None
    for line in reversed([line for line in lines if line]):
        try:
            shot_report = json.loads(line)
            break
        except json.JSONDecodeError:
            pass
    print("Shot report:", shot_report)


In [ ]:
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
manifest_path = artifact_root / "frame_extraction" / "manifests" / f"{os.environ['AIC_VIDEO_ID']}.jsonl"
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]

def timecode(seconds: float) -> str:
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

print("Records:", len(records))
print("Manifest:", manifest_path)
print("\nFull frame table:")
print("sample_n\ttimecode\tpts_time_s\tframe_idx\tfps\tkeyframe_n\tsource\timage_path")
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    print(
        f"{record['sample_n']}\t{timecode(record['pts_time_s'])}\t"
        f"{record['pts_time_s']:.6f}\t{record['frame_idx']}\t{record['fps']:.4f}\t"
        f"{record.get('keyframe_n')}\t{record['sampling_source']}\t{image_path}"
    )

thumbs = []
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    image = Image.open(image_path).convert("RGB")
    thumb = image.copy()
    thumb.thumbnail((240, 135))
    canvas = Image.new("RGB", (240, 160), "white")
    canvas.paste(thumb, ((240 - thumb.width) // 2, 0))
    draw = ImageDraw.Draw(canvas)
    draw.text(
        (8, 140),
        f"#{record['sample_n']} {timecode(record['pts_time_s'])}",
        fill=(0, 0, 0),
    )
    thumbs.append(canvas)

cols = 2
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 240, max(1, rows) * 160), "white")
for index, thumb in enumerate(thumbs):
    sheet.paste(thumb, ((index % cols) * 240, (index // cols) * 160))
display(Markdown("## Thumbnail overview"))
display(sheet)

display(Markdown("## Full extracted frames"))
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    display(Markdown(
        f"### Frame #{record['sample_n']} - {timecode(record['pts_time_s'])} "
        f"({record['pts_time_s']:.3f}s, frame_idx={record['frame_idx']})"
    ))
    display(Image.open(image_path).convert("RGB"))


## Organizer vs TransNetV2 comparison

This section preserves every organizer keyframe, adds adaptive TransNetV2 candidates outside the 0.5-second dedupe window, and renders comparison pairs. Set `AIC_COMPARE_PREVIEW_SELECTION=page`, `AIC_COMPARE_PREVIEW_PAGE=1`, and `AIC_COMPARE_PREVIEW_PAIRS=24` to browse additions page by page.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
video_id = os.environ["AIC_VIDEO_ID"]
shot_manifest_path = artifact_root / "shot_detection" / f"{video_id}.jsonl"
comparison_preview_pairs = int(os.environ.get("AIC_COMPARE_PREVIEW_PAIRS", "8"))
comparison_preview_selection = os.environ.get("AIC_COMPARE_PREVIEW_SELECTION", "evenly")
comparison_preview_page = int(os.environ.get("AIC_COMPARE_PREVIEW_PAGE", "1"))
comparison_command = [
    sys.executable,
    "-u",
    "scripts/compare_frame_sampling.py",
    "--video-id",
    video_id,
    "--map-csv",
    os.environ["AIC_MAP_CSV"],
    "--shots",
    str(shot_manifest_path),
    "--video-path",
    os.environ["AIC_VIDEO_PATH"],
    "--output-root",
    str(artifact_root),
    "--tolerance-s",
    "0.5",
    "--preview-pairs",
    str(comparison_preview_pairs),
    "--preview-selection",
    comparison_preview_selection,
    "--preview-page",
    str(comparison_preview_page),
]
print("$", " ".join(comparison_command), flush=True)
subprocess.run(comparison_command, check=True)


In [ ]:
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

comparison_dir = (
    Path(os.environ["AIC_ARTIFACT_ROOT"])
    / "frame_extraction"
    / "comparison"
    / os.environ["AIC_VIDEO_ID"]
)
summary = json.loads((comparison_dir / "summary.json").read_text(encoding="utf-8"))
additions = [
    json.loads(line)
    for line in (comparison_dir / "transnetv2_additions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
preview_pairs = [
    json.loads(line)
    for line in (comparison_dir / "preview_pairs.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

display(Markdown(
    "## Sampling comparison summary\n"
    f"| Metric | Value |\n|---|---:|\n"
    f"| TransNetV2 shots | {summary['shots']} |\n"
    f"| Organizer frames | {summary['organizer_frames']} |\n"
    f"| Adaptive candidates before dedupe | {summary['adaptive_candidates_before_dedupe']} |\n"
    f"| Adaptive candidates overlapping organizer | {summary['adaptive_candidates_overlapping_organizer']} |\n"
    f"| New TransNetV2 additions | {summary['transnetv2_additions_after_dedupe']} |\n"
    f"| Merged frames | {summary['merged_frames']} |\n"
    f"| Preview selection | {summary['preview_selection']} |\n"
    f"| Preview page | {summary['preview_page']} / {summary['preview_page_count']} |\n"
    f"| Organizer max temporal gap | {summary['organizer_gap_stats']['max_s']:.3f}s |\n"
    f"| Merged max temporal gap | {summary['merged_gap_stats']['max_s']:.3f}s |"
))
print("Summary artifact:", comparison_dir / "summary.json")
print("Full merged manifest:", comparison_dir / "merged_samples.jsonl")
print("Full additions manifest:", comparison_dir / "transnetv2_additions.jsonl")

print_limit = int(os.environ.get("AIC_COMPARE_PRINT_LIMIT", "100"))
shown = additions if print_limit == 0 else additions[:print_limit]
print(f"\nTransNetV2 additions table: showing {len(shown)}/{len(additions)}")
print("sample_n\ttimecode\tpts_time_s\tframe_idx\tshot_id\tshot_start_idx\tshot_end_idx")
for record in shown:
    print(
        f"{record['sample_n']}\t{timecode(record['pts_time_s'])}\t"
        f"{record['pts_time_s']:.6f}\t{record['frame_idx']}\t{record['shot_id']}\t"
        f"{record['shot_start_idx']}\t{record['shot_end_idx']}"
    )

display(Markdown("## Direct visual comparison: organizer vs new TransNetV2 frame"))
for pair in preview_pairs:
    original = pair["organizer"]
    adaptive = pair["transnetv2"]
    panels = []
    for label, sample, path_key in (
        ("ORGANIZER", original, "organizer_image"),
        ("TRANSNETV2 ADDITION", adaptive, "transnetv2_image"),
    ):
        image = Image.open(pair[path_key]).convert("RGB")
        image.thumbnail((360, 203))
        panel = Image.new("RGB", (360, 245), "white")
        panel.paste(image, ((360 - image.width) // 2, 0))
        draw = ImageDraw.Draw(panel)
        draw.text((8, 208), label, fill=(0, 0, 0))
        draw.text(
            (8, 225),
            f"{timecode(sample['pts_time_s'])} | frame_idx={sample['frame_idx']}",
            fill=(0, 0, 0),
        )
        panels.append(panel)
    row = Image.new("RGB", (720, 245), "white")
    row.paste(panels[0], (0, 0))
    row.paste(panels[1], (360, 0))
    display(Markdown(
        f"### Addition #{pair['addition_position']} - timestamp distance {pair['delta_s']:.3f}s"
    ))
    display(row)


## Export all adaptive additions to Google Drive

Create a Kaggle Secret named `AIC_GDRIVE_OAUTH_JSON`, enable it for this notebook, and store an OAuth refresh-token payload with Drive scope:

```json
{"client_id": "...apps.googleusercontent.com", "client_secret": "...", "refresh_token": "..."}
```

Authorize the refresh token with `https://www.googleapis.com/auth/drive.file`. The cell extracts every frame in `transnetv2_additions.jsonl`, creates a Drive folder, uploads images using canonical frame-index filenames, and reuses matching Drive files on reruns. Set `AIC_EXPORT_GDRIVE=0` to disable export.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

EXPORT_TO_GDRIVE = os.environ.get("AIC_EXPORT_GDRIVE", "1") == "1"
gdrive_oauth_json = ""
if EXPORT_TO_GDRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        gdrive_oauth_json = UserSecretsClient().get_secret("AIC_GDRIVE_OAUTH_JSON") or ""
    except Exception as error:
        print(f"Google Drive export skipped: Kaggle Secret AIC_GDRIVE_OAUTH_JSON is unavailable ({type(error).__name__}).")

if not EXPORT_TO_GDRIVE:
    print("Google Drive export disabled by AIC_EXPORT_GDRIVE=0.")
elif not gdrive_oauth_json:
    print("Add and enable Kaggle Secret AIC_GDRIVE_OAUTH_JSON, then rerun this cell.")
else:
    artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
    video_id = os.environ["AIC_VIDEO_ID"]
    additions_path = (
        artifact_root
        / "frame_extraction"
        / "comparison"
        / video_id
        / "transnetv2_additions.jsonl"
    )
    extract_command = [
        sys.executable,
        "-u",
        "scripts/extract_adaptive_frames.py",
        "--video-id",
        video_id,
        "--video-path",
        os.environ["AIC_VIDEO_PATH"],
        "--candidates",
        str(additions_path),
        "--output-root",
        str(artifact_root),
    ]
    print("$", " ".join(extract_command), flush=True)
    subprocess.run(extract_command, check=True)

    try:
        google_dependencies_available = all(
            importlib.util.find_spec(name) is not None
            for name in ("google.auth", "googleapiclient")
        )
    except ModuleNotFoundError:
        google_dependencies_available = False
    if not google_dependencies_available:
        install_command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "google-api-python-client",
            "google-auth",
            "google-auth-httplib2",
        ]
        print("Installing Google Drive API client...", flush=True)
        subprocess.run(install_command, check=True)

    upload_command = [
        sys.executable,
        "-u",
        "scripts/export_frame_artifacts_to_gdrive.py",
        "--video-id",
        video_id,
        "--output-root",
        str(artifact_root),
        "--folder-name",
        os.environ.get("AIC_GDRIVE_FOLDER_NAME", f"AIC2026-{video_id}-adaptive-frames"),
    ]
    upload_environment = os.environ.copy()
    upload_environment["AIC_GDRIVE_OAUTH_JSON"] = gdrive_oauth_json
    print("$", " ".join(upload_command), flush=True)
    subprocess.run(upload_command, check=True, env=upload_environment)
